# The Gulf of Specification: Why Your Eval Rubric Matters More Than Your Eval Framework

This notebook demonstrates a core problem in AI evaluation that is easy to miss:

**Two judges. Same model. Same responses. Completely different scores.**

The only difference is how precisely "good" was defined in the rubric.

This is the gulf of specification — the gap between what you intend your product to do and what you actually encode into your evaluation system.

When the spec is soft, the judge fills in the gaps with its own assumptions. When the spec is precise, the judge measures what you actually care about.

---

**What we test:**
- A customer support AI responding to user queries
- Three response types: genuinely good, fluent but wrong, and confidently off-topic
- Two judges: one with a vague rubric, one with a precise rubric
- Side-by-side score comparison to show where the spec gap creates false confidence

## Setup

In [ ]:
!pip install openai --quiet

In [ ]:
import os
import json
from openai import OpenAI
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## The Test Cases

Three customer support responses to the same query.

- **Response A**: Correct, specific, actionable
- **Response B**: Fluent, confident, factually wrong (classic quiet failure)
- **Response C**: Polite, helpful-sounding, completely misses the question

A good eval should clearly separate these. A vague one won't.

In [ ]:
user_query = "My payment failed but I was still charged. How do I get a refund?"

responses = {
    "A - Correct and specific": """
        I'm sorry this happened. If your payment failed but you were charged, 
        this is likely a temporary authorization hold. It should be released within 
        3-5 business days automatically. If it hasn't resolved by then, please contact 
        your bank directly with transaction ID and we can also raise a dispute on our end. 
        You can find your transaction ID in your account under Order History.
    """,
    "B - Fluent but wrong": """
        No problem! Failed payments are never actually charged to your account. 
        What you're seeing is just a display error in our system. 
        It will disappear within 24 hours and no money has left your account. 
        You don't need to take any action.
    """,
    "C - Polite but misses the point": """
        Thank you for reaching out to us! We really value your business and are 
        sorry to hear you're having trouble. Our payment system is very secure and 
        we take all issues seriously. Please don't hesitate to contact us again 
        if you need anything else. We're always here to help!
    """
}

print("User query:", user_query)
print("\nTest responses loaded:", list(responses.keys()))

## Judge 1: Vague Rubric

This is the kind of eval most teams start with.
Generic dimensions, no definitions, no pass/fail criteria.
The judge uses its own judgment to fill in what 'good' means.

In [ ]:
def judge_vague(query, response):
    prompt = f"""
You are evaluating a customer support response.

User query: {query}

Response: {response}

Score this response on the following dimensions (1-5):
- Helpfulness
- Accuracy  
- Clarity

Return JSON only:
{{
  "helpfulness": <score>,
  "accuracy": <score>,
  "clarity": <score>,
  "overall": <average>,
  "reasoning": "<one sentence>"
}}
"""
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    raw = result.choices[0].message.content.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

print("Vague judge defined.")

## Judge 2: Precise Rubric

Same model. Same query. But now each dimension has an explicit definition
and a clear pass/fail criteria tied to what actually matters for this product.

This is what closing the gulf of specification looks like in practice.

In [ ]:
def judge_precise(query, response):
    prompt = f"""
You are evaluating a customer support response for a payments product.

User query: {query}

Response: {response}

Score this response on the following dimensions (1-5) using the definitions below:

FACTUAL ACCURACY (1-5):
- 5: Every factual claim is correct. No false reassurances.
- 3: Mostly correct but missing key caveats.
- 1: Contains false information (e.g. claiming no charge occurred when one did).
FAIL if the response reassures the user that no money was taken when the query says they were charged.

QUESTION RESOLUTION (1-5):
- 5: Directly answers the specific question asked (how to get a refund).
- 3: Partially addresses the situation but misses the refund question.
- 1: Does not address the question at all. Generic or off-topic.
FAIL if the response contains no actionable step toward resolving the charge.

CLARITY (1-5):
- 5: User knows exactly what to do next.
- 3: Response is understandable but next steps are unclear.
- 1: User cannot determine what action to take from this response.

Return JSON only:
{{
  "factual_accuracy": <score>,
  "question_resolution": <score>,
  "clarity": <score>,
  "overall": <average>,
  "reasoning": "<one sentence>"
}}
"""
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    raw = result.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

print("Precise judge defined.")

## Run Both Judges

In [ ]:
results = {}

for label, response in responses.items():
    print(f"Evaluating: {label}")
    vague = judge_vague(user_query, response)
    precise = judge_precise(user_query, response)
    results[label] = {"vague": vague, "precise": precise}
    print(f"  Vague overall:   {vague['overall']} — {vague['reasoning']}")
    print(f"  Precise overall: {precise['overall']} — {precise['reasoning']}")
    print()

## Side-by-Side Comparison

In [ ]:
print(f"{'Response':<35} {'Vague Overall':>15} {'Precise Overall':>17} {'Gap':>6}")
print("-" * 75)

for label, scores in results.items():
    v = scores["vague"]["overall"]
    p = scores["precise"]["overall"]
    gap = round(v - p, 2)
    print(f"{label:<35} {str(v):>15} {str(p):>17} {str(gap):>6}")

print()
print("A positive gap means the vague judge scored it HIGHER than the precise judge.")
print("The larger the gap on B and C, the more the vague spec created false confidence.")

## What This Shows

The vague judge scores Response B (fluent but factually wrong) similarly to Response A (correct and specific).
It sees confident, well-structured text and rewards it — because 'accuracy' without a definition defaults to surface fluency.

The precise judge catches Response B immediately because the rubric explicitly says:
*FAIL if the response reassures the user that no money was taken when the query says they were charged.*

Response C — polite but useless — scores high on the vague judge's 'helpfulness' dimension
because it sounds caring. The precise judge scores it low on 'question resolution' because
it contains no actionable step toward a refund.

---

**The takeaway:**

Both judges use the same model. The difference is entirely in the definition of good.
This is the gulf of specification — and it shows up in every eval that uses generic dimensions
without application-specific criteria.

Evals are only as honest as the definitions behind them.

## Further Exploration

Try modifying the rubric and observe how scores shift:

1. Add a new test case where the response is technically correct but uses jargon the user won't understand — does the vague judge catch it?
2. Make the precise rubric even more specific — add a FAIL condition for responses longer than 150 words. Does the overall score change?
3. Run the same judges with `temperature=0.7` instead of `0.1` — how much does score variance increase between runs?